# Trois outils, un problème

| Module | Durée | Prérequis |
|---|---|---|
| M6 – Orthogonalité et moindres carrés | 75 min | M5 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Une start-up fabrique une station météo à bas prix. Son capteur de température renvoie une **tension** (en millivolts) qui dépend de la température, mais pas de façon linéaire. Pour l'étalonner, on a placé le capteur dans une enceinte réglée de 20 °C à 80 °C, et relevé 61 tensions. On cherche un polynôme qui donne la tension en fonction de la température, par moindres carrés.

Trois stagiaires font le calcul, chacun avec son outil :

- **Noé** résout les équations normales $A^{\mathsf{T}}A\,\hat{\mathbf{x}} = A^{\mathsf{T}}\mathbf{b}$ avec `np.linalg.solve` ;
- **Inaya** utilise directement `np.linalg.lstsq` ;
- **Sacha** utilise la décomposition **QR**, que l'ingénieure lui a recommandée.

Pour un polynôme de degré 3, ils trouvent la même chose. Pour un degré 8, leurs résultats ne concordent plus.

**Les questions : pourquoi trois méthodes mathématiquement équivalentes donnent-elles des résultats différents ? Laquelle choisir, et selon quels critères ?**

**Activité** : Quels critères peuvent compter pour choisir un outil de calcul ? Pensez à la précision, mais pas seulement.

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_plus_precis = "Noé"          # "Noé", "Inaya" ou "Sacha" (modifiez)
pari_plus_rapide = "Sacha"        # (modifiez)

## ③ Abstraire : trois chemins vers la même projection

Les trois méthodes cherchent le même $\hat{\mathbf{x}}$ : celui qui rend $\lVert A\mathbf{x} - \mathbf{b} \rVert$ minimal.

**Les équations normales.** On forme la petite matrice carrée $A^{\mathsf{T}}A$ et on résout. C'est rapide, mais le conditionnement est **élevé au carré** :

$$
\operatorname{cond}(A^{\mathsf{T}}A) = \operatorname{cond}(A)^2 .
$$

Au module M4, on a vu que le conditionnement mesure l'amplification des erreurs. Un conditionnement de $10^8$ pour $A$ devient $10^{16}$ : toute la précision de l'ordinateur (environ 16 chiffres) est perdue.

**La décomposition QR.** On écrit $A = QR$, où les colonnes de $Q$ sont **orthonormées** (de longueur 1 et deux à deux orthogonales) et $R$ est triangulaire supérieure. On les obtient par le procédé de **Gram-Schmidt** : on redresse les colonnes une à une, en retirant à chacune sa projection sur les précédentes.

![Schéma : Gram-Schmidt ; q1 est a1 normalisé ; on retire de a2 sa projection sur q1 puis on normalise pour obtenir q2](../../assets/schemas/schema-gram-schmidt.svg)

Comme $Q^{\mathsf{T}}Q = I$, les équations normales deviennent $R^{\mathsf{T}}Q^{\mathsf{T}}QR\,\hat{\mathbf{x}} = R^{\mathsf{T}}Q^{\mathsf{T}}\mathbf{b}$, c'est-à-dire

$$
R\,\hat{\mathbf{x}} = Q^{\mathsf{T}}\mathbf{b} ,
$$

un système triangulaire, qu'on résout par remontée, comme à la fin du pivot de Gauss (module M2). Le conditionnement de $R$ est celui de $A$ : il n'est pas élevé au carré.

**`np.linalg.lstsq`** utilise une troisième décomposition, la SVD (module M8). Elle est la plus robuste, mais elle a une particularité : par défaut, elle **ignore les directions jugées négligeables**, comme la tolérance du rang au module M5.

**Activité** : Appliquez Gram-Schmidt aux colonnes $\mathbf{a}_1 = (3\,;\,4)$ et $\mathbf{a}_2 = (1\,;\,2)$. Vérifiez que $\mathbf{q}_1 = (0{,}6\,;\,0{,}8)$ et $\mathbf{q}_2 = (-0{,}8\,;\,0{,}6)$, puis écrivez $R$.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Un cas d'école, à la taille de la main

Avant le capteur, un exemple construit exprès pour révéler le problème. Avec $\varepsilon = 10^{-8}$ :

$$
A = \begin{pmatrix} 1 & 1 \\ \varepsilon & 0 \\ 0 & \varepsilon \end{pmatrix}, \qquad
\mathbf{b} = \begin{pmatrix} 2 \\ \varepsilon \\ \varepsilon \end{pmatrix}, \qquad
\hat{\mathbf{x}} = \begin{pmatrix} 1 \\ 1 \end{pmatrix} \text{ (solution exacte)}.
$$

**Activité** : Calculez $A^{\mathsf{T}}A$ à la main. Que devient $1 + \varepsilon^2$ pour un ordinateur qui garde 16 chiffres significatifs ?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

eps = 1e-8
A = np.array([[1, 1], [eps, 0], [0, eps]])
b = np.array([2, eps, eps])

print("A^T A =\n", A.T @ A)
print("1 + eps² == 1 pour l'ordinateur ?", 1 + eps ** 2 == 1)
print("cond(A) =", f"{np.linalg.cond(A):.1e}", "  cond(A^T A) =", f"{np.linalg.cond(A.T @ A):.1e}")

try:
    print("Équations normales :", np.linalg.solve(A.T @ A, A.T @ b))
except np.linalg.LinAlgError as erreur:
    print("Équations normales : échec ->", erreur)
Q, R = np.linalg.qr(A)
print("QR                 :", np.linalg.solve(R, Q.T @ b))
print("lstsq              :", np.linalg.lstsq(A, b, rcond=None)[0])

### Gram-Schmidt et la remontée, écrits à la main

On programme soi-même les deux étapes, pour voir qu'il n'y a pas de magie, puis on compare à `np.linalg.qr`.

In [ ]:
def gram_schmidt(A):
    A = np.asarray(A, dtype=float)
    m, n = A.shape
    Q, R = np.zeros((m, n)), np.zeros((n, n))
    for j in range(n):
        v = A[:, j].copy()
        for i in range(j):
            R[i, j] = Q[:, i] @ A[:, j]        # composante de a_j sur q_i
            v -= R[i, j] * Q[:, i]             # on la retire
        R[j, j] = np.linalg.norm(v)
        Q[:, j] = v / R[j, j]
    return Q, R

def remonter(R, y):
    # Résout R x = y quand R est triangulaire supérieure (de la dernière ligne vers la première)
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - R[i, i + 1:] @ x[i + 1:]) / R[i, i]
    return x

M = np.array([[3., 1], [4, 2]])
Qm, Rm = gram_schmidt(M)
print("Q =\n", Qm.round(4), "\nR =\n", Rm.round(4))
print("QR = M ?", np.allclose(Qm @ Rm, M), "  Q^T Q = I ?", np.allclose(Qm.T @ Qm, np.eye(2)))
Qn, Rn = np.linalg.qr(M)
print("np.linalg.qr donne les mêmes, au signe près :\n", Qn.round(4))

### L'étalonnage du capteur

In [ ]:
rng = np.random.default_rng(3)
T = np.linspace(20, 80, 61)                                      # températures de l'enceinte (°C)
tension = 3300 / (1 + np.exp(-(T - 50) / 12)) + rng.normal(0, 2, T.size)   # mesures (mV)

def trois_outils(A, b):
    resultats = {}
    try:
        resultats["Noé (équations normales)"] = np.linalg.solve(A.T @ A, A.T @ b)
    except np.linalg.LinAlgError:
        resultats["Noé (équations normales)"] = np.full(A.shape[1], np.nan)
    x, _, rang, _ = np.linalg.lstsq(A, b, rcond=None)
    resultats[f"Inaya (lstsq, rang vu : {rang})"] = x
    Q, R = np.linalg.qr(A)
    resultats["Sacha (QR)"] = remonter(R, Q.T @ b)
    return resultats

degre = 8
A_capteur = np.vander(T, degre + 1, increasing=True)
print(f"Degré {degre} : cond(A) = {np.linalg.cond(A_capteur):.1e}")

TT = np.linspace(20, 80, 400)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(T, tension, "o", ms=3, color=LIOUAI["gris"], label="mesures")
for (nom, x), couleur in zip(trois_outils(A_capteur, tension).items(), [LIOUAI["jaune"], LIOUAI["navy"], LIOUAI["vert"]]):
    courbe = np.vander(TT, degre + 1, increasing=True) @ x
    axes[0].plot(TT, courbe, color=couleur, lw=1.8, label=nom)
    axes[1].plot(T, tension - A_capteur @ x, color=couleur, lw=1.2, label=nom)
axes[0].set_xlabel("température (°C)"); axes[0].set_ylabel("tension (mV)"); axes[0].legend(fontsize=8)
axes[0].set_title(f"Polynôme de degré {degre}, trois outils")
axes[1].set_xlabel("température (°C)"); axes[1].set_ylabel("résidu (mV)"); axes[1].set_title("Résidus")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Comparer systématiquement

Pour juger, il faut une référence. La meilleure somme de carrés possible, pour chaque degré, est calculée avec une variable **remise à l'échelle** $s = (T - 50)/30$, qui va de $-1$ à $1$ (module M5) : le problème est alors bien conditionné et tous les outils sont d'accord. On mesure ensuite de combien chaque outil, en variable brute $T$, dépasse cette somme minimale.

In [ ]:
def exces(A, b, x, s_min):
    return np.sum((b - A @ x) ** 2) / s_min - 1     # 0 = parfait

s = (T - 50) / 30
print(f"{'degré':>5s} {'cond(A)':>9s}   {'Noé':>9s} {'Inaya':>9s} {'Sacha':>9s}")
for d in range(3, 11):
    A_d = np.vander(T, d + 1, increasing=True)
    A_s = np.vander(s, d + 1, increasing=True)
    s_min = np.sum((tension - A_s @ np.linalg.lstsq(A_s, tension, rcond=None)[0]) ** 2)
    ex = [exces(A_d, tension, x, s_min) for x in trois_outils(A_d, tension).values()]
    print(f"{d:5d} {np.linalg.cond(A_d):9.1e}   " + " ".join(f"{e:9.1e}" for e in ex))

**Activité** : Lisez le tableau. À partir de quel degré chaque outil décroche-t-il ? Pour Inaya, que signifie le « rang vu » affiché dans la légende du graphique ?

Indiquez vos réponses et justifiez-les.

Noé s'en sort mieux sur la **courbe** que sur les **coefficients**. Comparons ses coefficients à ceux de Sacha :

In [ ]:
for d in (6, 8, 10):
    A_d = np.vander(T, d + 1, increasing=True)
    res = list(trois_outils(A_d, tension).values())
    ecart = np.max(np.abs(res[0] - res[2]) / np.abs(res[2]))
    print(f"degré {d:2d} : écart relatif maximal entre les coefficients de Noé et de Sacha = {ecart:.1e}")

Comme au notebook 5.3, des coefficients assez différents peuvent décrire presque la même courbe : les erreurs se compensent. Mais dès qu'on a besoin des coefficients eux-mêmes (pour les programmer dans le capteur, par exemple), la perte de précision des équations normales devient gênante.

### Le coût en temps

La précision n'est pas le seul critère. Sur un grand problème (200 000 mesures, 20 inconnues), on chronomètre les trois outils.

In [ ]:
grand_A = rng.normal(size=(200_000, 20))
grand_b = rng.normal(size=200_000)

def chrono(f, repetitions=3):
    temps = []
    for _ in range(repetitions):
        debut = time.perf_counter(); f(); temps.append(time.perf_counter() - debut)
    return min(temps)

outils = {
    "équations normales": lambda: np.linalg.solve(grand_A.T @ grand_A, grand_A.T @ grand_b),
    "lstsq": lambda: np.linalg.lstsq(grand_A, grand_b, rcond=None),
    "QR": lambda: np.linalg.qr(grand_A),
}
for nom, f in outils.items():
    print(f"{nom:20s} : {1000 * chrono(f):7.1f} ms")

## ⑤ Interpréter

In [ ]:
print(f"Vos paris : plus précis = {pari_plus_precis}, plus rapide = {pari_plus_rapide}")

**Activité** :

1. Expliquez à l'équipe, en quelques phrases, pourquoi les trois stagiaires n'ont pas trouvé la même chose au degré 8.
2. Remplissez un tableau de choix : pour chaque outil, ses points forts, ses points faibles, et une situation où vous le choisiriez.
3. Le vrai problème, ici, est-il l'outil ou la **formulation** ? Que change la remise à l'échelle de la température ?
4. Un polynôme de degré 8 est-il de toute façon un bon choix pour étalonner ce capteur ? Le notebook 6.4 reviendra sur cette question.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Les sources d'erreur.** Faites la liste de toutes les erreurs qui séparent la courbe calculée de la « vraie » réponse du capteur : bruit de mesure, choix du modèle, arrondis de l'ordinateur, tolérance des outils. Lesquelles dominent ici ?
- **Vérifier autrement.** Pour le degré 3, tous les outils sont d'accord. Ce constat suffit-il à prouver qu'ils sont justes ? Comment l'avez-vous vérifié dans ce notebook ?
- **Généraliser.** Le procédé de Gram-Schmidt programmé plus haut est connu pour perdre de l'orthogonalité quand les colonnes sont presque dépendantes. Mesurez $\lVert Q^{\mathsf{T}}Q - I \rVert$ pour la matrice du capteur, avec votre fonction et avec `np.linalg.qr`, pour les degrés 3 à 10. Concluez.

In [ ]:
def perte_orthogonalite(Q):
    # À compléter : renvoie la norme de Q^T Q - I
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def perte_orthogonalite(Q):
    return np.linalg.norm(Q.T @ Q - np.eye(Q.shape[1]))

print(f"{'degré':>5s} {'Gram-Schmidt maison':>20s} {'np.linalg.qr':>14s}")
for d in range(3, 11):
    A_d = np.vander(T, d + 1, increasing=True)
    print(f"{d:5d} {perte_orthogonalite(gram_schmidt(A_d)[0]):20.1e} {perte_orthogonalite(np.linalg.qr(A_d)[0]):14.1e}")
print("np.linalg.qr utilise une autre méthode (réflexions de Householder), qui garde l'orthogonalité.")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Comparer les mérites des outils dans le contexte": 1,
    "Connaître des outils comparables": 1,
    "Lister les sources d'erreur liées aux limites du calcul": 1,
    "Connaître les outils associés au concept": 1,
    "Corroborer ses résultats": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "6.3-trois-outils", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")